In [1]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
import os

In [2]:
figure_dir = "../results/figures"
table_dir = "../results/tables"

In [4]:
adata = sc.read_h5ad(os.path.expanduser("../data/processed/04_spagcn.h5ad"))

In [5]:
bs_results = pd.read_csv(os.path.join(table_dir, "bayesspace_clusters.csv"), index_col=0)

In [6]:
adata.obs["bayesspace"] = bs_results.loc[adata.obs.index, "bayesspace_cluster"].astype(str).astype("category")

In [7]:
gt = adata.obs["ground_truth"]
methods = {
    "Leiden": adata.obs["leiden"],
    "SpaGCN": adata.obs["spagcn_refined"],
    "BayesSpace": adata.obs["bayesspace"]
}

In [8]:
results = []
for name, pred in methods.items():
    mask = gt.notna() & pred.notna()
    ari = adjusted_rand_score(gt[mask], pred[mask])
    nmi = normalized_mutual_info_score(gt[mask], pred[mask])
    results.append({"Method": name, "ARI": f"{ari:.4f}", "NMI": f"{nmi:.4f}"})

In [9]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(table_dir, "clustering_comparison.csv"), index=False)

    Method    ARI    NMI
    Leiden 0.3882 0.5009
    SpaGCN 0.3163 0.4593
BayesSpace 0.5355 0.6729


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

sc.pl.spatial(adata, color="ground_truth", ax=axes[0, 0], show=False,
              title="Ground Truth", spot_size=100)
sc.pl.spatial(adata, color="leiden", ax=axes[0, 1], show=False,
              title=f"Leiden\nARI={results[0]['ARI']}", spot_size=100)
sc.pl.spatial(adata, color="spagcn_refined", ax=axes[1, 0], show=False,
              title=f"SpaGCN\nARI={results[1]['ARI']}", spot_size=100)
sc.pl.spatial(adata, color="bayesspace", ax=axes[1,1], show=False,
              title=f"BayesSpace\nARI={results[2]['ARI']}", spot_size=100)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "clustering_comparison_spatial.png"),
            dpi=200, bbox_inches='tight')
plt.close()

/tmp/ipykernel_38083/779058199.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="ground_truth", ax=axes[0, 0], show=False,
/tmp/ipykernel_38083/779058199.py:5: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="leiden", ax=axes[0, 1], show=False,
/tmp/ipykernel_38083/779058199.py:7: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="spagcn_refined", ax=axes[1, 0], show=False,
/tmp/ipykernel_38083/779058199.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="bayesspace", ax=axes[1,1], show=False,


In [13]:
adata.write(os.path.join("../data/processed/04_all_clustering.h5ad"))